# 🎯 [ISR02_M01] 전장 SNS(텔레그램, X 등) 텍스트/이미지 게시물 수집 및 OSINT 원천 데이터셋 정의
### Google Colab 실습 환경 (정보·감시·정찰(ISR) / Level 2)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/ISR02/isr02_m01_student.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 정보·감시·정찰(ISR) (Level 2) |
| **미션 ID** | ISR02_M01 |
| **미션 주제** | 전장 SNS(텔레그램, X 등) 텍스트/이미지 게시물 수집 및 OSINT 원천 데이터셋 정의 |
| **소요 시간** | 약 1.5시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
러시아-우크라이나 전쟁에서는 텔레그램·X 같은 SNS에 현장 제보와 함께 재게시, 수치·지역 변조, 과거 사진 재활용, 생성형 AI 조작물, 조직적 동시 유포가 한꺼번에 쏟아지면서 공개출처정보(OSINT)의 출처와 신뢰도를 빠르게 가려내는 일이 정보전의 핵심이 되었다. 본 프로젝트에서는 가상 계정과 가상 지역 코드로 만든 교육용 합성 게시물을 사용해 원천 데이터 스키마 정의, 텍스트 정제와 이미지 지각 해시 기반 중복 제거, 유사도 기반 복제·1차 출처 분류 Baseline, 조작·기만 탐지와 계정 신뢰도 결합 모델, 종합 신뢰도 등급(A~E) 엔진, 정보전 모니터링 대시보드, 정보관 긴급 검토(HITL), 대량 스트림 부하 시험, AAR 재보정, 최종 통합까지 단계별로 구현한다. 최종 목표는 조작 게시물을 놓치지 않으면서 사실 제보를 가짜로 몰지 않고, 파급력이 큰 첩보의 채택·차단은 원천 출처를 확인한 정보관이 결정하도록 설계하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

## 📂 2. 산출물 디렉토리 체계 및 실습 가이드

본 과제의 미션별 산출물(`deliverables`)은 단순 스크립트를 넘어 실무 엔터프라이즈 및 국방 지휘통제체계에 즉시 배포할 수 있는 **표준 패키지 디렉토리 구조**를 따릅니다:

| 디렉토리 | 주요 산출물(Deliverables) | 역할 및 설명 |
| :--- | :--- | :--- |
| **`data/`** | `data/schema/isr02_input_schema.json` | 입출력 데이터 표준 스키마 및 원천/정제 데이터셋 |
| **`src/`** | `src/preprocess/`, `src/models/`, `src/verification/`, `src/ui/`, `src/hitl/` | 전처리, Baseline, 앙상블 탐지, 대시보드, HITL 등 핵심 파이썬 소스 코드 모듈 |
| **`docs/`** | `docs/isr02_data_spec.md`, `docs/isr02_hitl_guide.md`, `docs/isr02_aar_report.md`, `docs/isr02_final_report.md` | 데이터 규격서, HITL 가이드, AAR 분석 보고서, 최종 브리핑 등 군사 행정/지휘관 문서 |
| **`configs/`**| `configs/isr02_tuned_params.yaml` | AAR 피드백 기반 재조정 임계치 및 환경설정 파일 |
| **`tests/`** | `tests/isr02_stress_test.py` | 500% 부하 시험 및 단위/통합 테스트 스위트 |

> **💡 [Google Colab / 주피터 노트북 실습 팁]**
> - **방법 A. 자동 파일 저장 (권장):** 코드 셀 첫 줄에 `%%writefile src/preprocess/isr02_preprocessor.py`와 같이 매직 명령어를 추가하여 실행하면, 노트북에서 작업한 코드가 해당 폴더의 파일로 자동 추출·저장됩니다.
> - **방법 B. 논리적 모듈 대응:** 별도 파일 추출 없이 노트북 셀 단위로 순차 실행하더라도 각 미션이 해당 모듈의 역할을 1:1로 수행하며, **미션 10(시스템 통합)**에서 전체 파이프라인이 유기적으로 완성되도록 설계되어 있습니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 01. 전장 SNS(텔레그램, X 등) 텍스트/이미지 게시물 수집 및 OSINT 원천 데이터셋 정의
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

텔레그램, X 등 전장 SNS에서 수집된 텍스트 및 이미지 게시물 데이터 구조를 분석하고 원천 데이터셋 스키마를 정의한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **텔레그램, X 등 전장 SNS에서 수집한 텍스트·이미지 게시물의 구조를 분석하고, 수집 경위와 주장 내용까지 담은 OSINT 원천 데이터셋 스키마와 검증기를 만든다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **게시물 기본 정보(게시물·계정 ID, 플랫폼, 게시 시각, 본문, 언어), 미디어(종류, 크기, 지각 해시), 전달 원본(forwarded_from), 수집 경위(수집 방식·시각·수집자·수집 지연), 주장 내용(가상 지역 코드, 위치 불확실성), 평가 상태 필드 정의하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `data/schema/isr02_input_schema.json 및 docs/isr02_data_spec.md` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 정보·감시·정찰(ISR) 실무 및 러시아-우크라이나 전쟁 전장 환경과의 군사적 부합성

#### 미션 PASS 조건

- 정량 기준: 스키마 필드 정의 완성도 100%, 샘플 데이터 검증 오류 0건


- 진행 기준: 입력 데이터 유효성 검증 스크립트 통과 및 명세서 문서화 완료


- 검증 방법: JSON Schema 검증 도구 및 샘플 데이터 파싱 단위 테스트


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""전장 SNS(텔레그램, X 등) 텍스트·이미지 게시물의 수집 메타데이터, 계정·전달 경로, 주장 내용(지역·시각), 수집 지연 등 OSINT 원천 데이터셋 스키마를 정의하고 가상 샘플 게시물을 검증한다."""
import json
import re
import numpy as np
import pandas as pd

ISO = r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}Z$"
# 1) 스키마 정의: 게시물 기본 정보, 미디어, 수집 경위, 주장 내용, 수집 조건(위치 불확실성·지연), 평가 상태
SCHEMA = {
    "required": ["post_id", "platform", "account_id", "posted_at", "text", "media", "collection", "claim", "assessment"],
    "fields": {
        "post_id": {"type": str, "pattern": r"^P\d{4}$"},
        "platform": {"type": str, "enum": ["telegram", "x", "other"]},
        "account_id": {"type": str, "pattern": r"^ACC_[A-Z]{3}\d{2}$"},
        "posted_at": {"type": str, "pattern": ISO},
        "text": {"type": str, "minlen": 1, "maxlen": 4000},
        "lang": {"type": str, "enum": ["ko", "en", "uk", "ru", "other"]},
    },
    "media": {
        "type": {"type": str, "enum": ["image", "video"]},
        "width": {"type": int, "min": 1, "max": 10000},
        "height": {"type": int, "min": 1, "max": 10000},
        "phash": {"type": str, "pattern": r"^[0-9a-f]{16}$"},
    },
    "collection": {
        "method": {"type": str, "enum": ["public_api", "manual_capture", "partner_share"]},
        "collected_at": {"type": str, "pattern": ISO},
        "collector": {"type": str, "pattern": r"^COL-\d{2}$"},
        "delay_min": {"type": (int, float), "min": 0, "max": 100000},
    },
    "claim": {
        "region_code": {"type": str, "pattern": r"^[A-F]-\d$"},
        "geo_uncertainty_km": {"type": (int, float), "min": 0, "max": 500},
    },
    # 수집 단계에서는 진위·출처 판정을 적지 않는다(정보관 검토 후 별도 기록)
    "assessment": {"status": {"type": str, "enum": ["unassessed"]}},
}


def check(name, value, rule):
    # [TODO : 학생 실습 1] 전장 SNS(텔레그램, X 등) 텍스트/이미지 게시물 수집 및 OSINT 원천 데이터셋 정의 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def validate(rec):
    # [TODO : 학생 실습 1] 전장 SNS(텔레그램, X 등) 텍스트/이미지 게시물 수집 및 OSINT 원천 데이터셋 정의 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
rng = np.random.default_rng(1)
posts = []
for i in range(20):
    r = f"{'ABCDEF'[int(rng.integers(6))]}-{int(rng.integers(1, 6))}"
    posts.append({"post_id": f"P{i:04d}", "platform": str(rng.choice(["telegram", "x", "other"])),
                  "account_id": f"ACC_{rng.choice(['MED', 'WIT', 'AGG', 'PRO'])}{int(rng.integers(0, 30)):02d}",
                  "posted_at": f"2026-05-0{int(rng.integers(1, 9))}T1{int(rng.integers(0, 9))}:0{int(rng.integers(0, 9))}Z",
                  "text": f"가상 {r} 지역에서 군용 차량 목격, {int(rng.integers(2, 15))}대", "lang": "ko",
                  "media": [{"type": "image", "width": 32, "height": 32, "phash": "".join(rng.choice(list("0123456789abcdef"), 16))}],
                  "forwarded_from": f"P{int(rng.integers(0, max(i, 1))):04d}" if i and rng.random() < .4 else None,
                  "collection": {"method": str(rng.choice(["public_api", "manual_capture", "partner_share"])), "collected_at": "2026-05-09T12:00Z",
                                 "collector": f"COL-{int(rng.integers(1, 20)):02d}", "delay_min": int(rng.integers(1, 600))},
                  "claim": {"region_code": r, "geo_uncertainty_km": float(round(rng.uniform(0.5, 20), 1))},
                  "assessment": {"status": "unassessed"}})

# 3) 정상 데이터 검증
ok = sum(1 for p in posts if not validate(p))
print(f"정상 샘플 {len(posts)}건 중 검증 통과 {ok}건, 오류 {len(posts) - ok}건 (목표: 오류 0건 {'달성' if ok == len(posts) else '미달'})")

# 4) 오류 주입 시험
bad = json.loads(json.dumps(posts[3]))
bad["platform"] = "unknown_app"
bad["posted_at"] = "2026/05/01 10:00"
del bad["collection"]["method"]
bad["media"][0]["phash"] = "XYZ"
bad["text"] = ""
print("\n[오류 주입 결과] 주입 5종")
for e in validate(bad):
    print(" -", e)

# 5) 스키마 필드 정의서(명세서 초안)
rows = [("기본", k, r["type"].__name__, str({x: y for x, y in r.items() if x != "type"})) for k, r in SCHEMA["fields"].items()]
for g in ("media", "collection", "claim", "assessment"):
    for k, r in SCHEMA[g].items():
        t = r["type"].__name__ if isinstance(r["type"], type) else "number"
        rows.append((g, k, t, str({x: y for x, y in r.items() if x != "type"})))
spec = pd.DataFrame(rows, columns=["그룹", "필드", "자료형", "규칙"])
print("\n[데이터 명세 요약]")
print(spec.to_string(index=False))
print(f"\n필드 정의 {len(spec)}개 + forwarded_from(전달 원본, 선택). 수집 지연·위치 불확실성은 '통신지연·위치오차' 전술 파라미터에 해당한다.")
print(f"전달 경로가 있는 게시물 {sum(p['forwarded_from'] is not None for p in posts)}건: 출처 역추적(미션 3·7)에 사용")


<details>
<summary><b>💡 [참고] 미션 01 예상 실행 결과 (Output)</b></summary>

```text
정상 샘플 20건 중 검증 통과 20건, 오류 0건 (목표: 오류 0건 달성)

[오류 주입 결과] 주입 5종
 - platform: 허용되지 않은 값 'unknown_app'
 - posted_at: 형식 불일치 '2026/05/01 10:00'
 - text: 빈 값
 - collection.method: 필수 필드 누락
 - media[0].phash: 형식 불일치 'XYZ'

[데이터 명세 요약]
        그룹                 필드    자료형                                                          규칙
        기본            post_id    str                                    {'pattern': '^P\\d{4}$'}
        기본           platform    str                        {'enum': ['telegram', 'x', 'other']}
        기본         account_id    str                         {'pattern': '^ACC_[A-Z]{3}\\d{2}$'}
        기본          posted_at    str        {'pattern': '^\\d{4}-\\d{2}-\\d{2}T\\d{2}:\\d{2}Z$'}
        기본               text    str                               {'minlen': 1, 'maxlen': 4000}
        기본               lang    str                 {'enum': ['ko', 'en', 'uk', 'ru', 'other']}
     media               type    str                                {'enum': ['image', 'video']}
     media              width    int                                    {'min': 1, 'max': 10000}
     media             height    int                                    {'min': 1, 'max': 10000}
     media              phash    str                               {'pattern': '^[0-9a-f]{16}$'}
collection             method    str {'enum': ['public_api', 'manual_capture', 'partner_share']}
collection       collected_at    str        {'pattern': '^\\d{4}-\\d{2}-\\d{2}T\\d{2}:\\d{2}Z$'}
collection          collector    str                                 {'pattern': '^COL-\\d{2}$'}
collection          delay_min number                                   {'min': 0, 'max': 100000}
     claim        region_code    str                                  {'pattern': '^[A-F]-\\d$'}
     claim geo_uncertainty_km number                                      {'min': 0, 'max': 500}
assessment             status    str                                    {'enum': ['unassessed']}

필드 정의 17개 + forwarded_from(전달 원본, 선택). 수집 지연·위치 불확실성은 '통신지연·위치오차' 전술 파라미터에 해당한다.
전달 경로가 있는 게시물 12건: 출처 역추적(미션 3·7)에 사용
```
</details>
